# BrandSphere AI — Week 4: Slogan Engine (NLP)
**CRS AI Capstone 2025–26**

Covers:
1. Text preprocessing with NLTK
2. TF-IDF vectorisation on slogan corpus + startups taglines
3. Cosine similarity retrieval
4. Brand tone analysis
5. Gemini API integration for creative generation

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly nltk Pillow 2>/dev/null
import subprocess, os, sys
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
import os; os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import nltk, re, warnings; warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('✅ Libraries loaded')

## 1. Load Datasets

In [ ]:
df_sl = pd.read_csv('datasets/processed/cleaned_slogans.csv')
df_st = pd.read_csv('datasets/raw/startups.csv')

# Combine slogan corpus: brand slogans + startup taglines
taglines = df_sl['Slogan'].dropna().tolist()
startup_tags = df_st['tagline'].dropna().sample(500, random_state=42).tolist()
corpus = taglines + startup_tags

print(f'Brand slogans:     {len(taglines)}')
print(f'Startup taglines:  {len(startup_tags)}')
print(f'Total corpus:      {len(corpus)}')
print(f'\nSample slogans:')
for s in taglines[:8]:
    print(f'  • {s}')

## 2. NLTK Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text  = str(text).lower()
    text  = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Analyse the real brand slogans
df_sl['preprocessed'] = df_sl['Slogan'].apply(preprocess)
df_sl['word_count']   = df_sl['Slogan'].apply(lambda x: len(str(x).split()))
df_sl['char_count']   = df_sl['Slogan'].apply(len)

print('Preprocessing results:')
print(df_sl[['Company','Slogan','preprocessed','word_count']].to_string())

## 3. TF-IDF Vectorisation

In [ ]:
# Build TF-IDF matrix on full corpus
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=1)
corpus_preprocessed = [preprocess(t) for t in corpus]
tfidf_matrix = vectorizer.fit_transform(corpus_preprocessed)

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'\nTop 20 terms by IDF (most discriminative):')
idf_scores = vectorizer.idf_
feature_names = vectorizer.get_feature_names_out()
top_idx = np.argsort(idf_scores)[-20:][::-1]
for idx in top_idx:
    print(f'  {feature_names[idx]:25s}: {idf_scores[idx]:.3f}')

## 4. TF-IDF Retrieval Function

In [ ]:
def retrieve_slogans(query: str, top_k: int = 5) -> list:
    """Retrieve most similar slogans to a query using cosine similarity."""
    q_vec  = vectorizer.transform([preprocess(query)])
    sims   = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idx:
        if sims[idx] > 0.01:
            results.append({'text': corpus[idx], 'score': round(float(sims[idx]), 4)})
    return results

# Test retrieval
test_queries = [
    'innovative technology solutions',
    'luxury fashion premium quality',
    'healthy food sustainable living',
    'financial freedom investment',
]
for query in test_queries:
    results = retrieve_slogans(query, top_k=3)
    print(f'\nQuery: "{query}"')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["text"]}')

## 5. NLTK Brand Analysis

In [ ]:
from nltk.tokenize import word_tokenize
from collections import Counter

def analyze_brand_text(text: str) -> dict:
    """Analyse brand text for tone, complexity, and key terms."""
    tokens    = word_tokenize(str(text).lower())
    words     = [t for t in tokens if t.isalpha()]
    stop_free = [w for w in words if w not in stop_words]
    
    avg_len   = np.mean([len(w) for w in words]) if words else 0
    unique_r  = len(set(words)) / max(len(words), 1)
    
    # Simple tone heuristics
    bold_words  = {'bold','powerful','strong','leader','dominate','pioneer','disrupt'}
    calm_words  = {'gentle','simple','clear','easy','quiet','soft','smooth'}
    action_words= {'achieve','build','create','drive','grow','lead','launch','win'}
    
    tone = 'Neutral'
    if any(w in bold_words for w in stop_free):   tone = 'Bold'
    elif any(w in calm_words for w in stop_free): tone = 'Calm'
    elif any(w in action_words for w in stop_free): tone = 'Action-Oriented'
    
    return {
        'word_count': len(words),
        'unique_ratio': round(unique_r, 3),
        'avg_word_length': round(avg_len, 2),
        'top_terms': [w for w,_ in Counter(stop_free).most_common(3)],
        'tone': tone,
    }

print('Brand analysis on real slogans:')
for _, row in df_sl.iterrows():
    analysis = analyze_brand_text(row['Slogan'])
    print(f"\n  {row['Company']}: '{row['Slogan']}'")
    for k,v in analysis.items():
        print(f'    {k}: {v}')

## 6. Visualise TF-IDF Top Terms

In [ ]:
# Show top TF-IDF terms across the brand slogan corpus
brand_corpus_joined = ' '.join([preprocess(s) for s in taglines])
brand_vec = TfidfVectorizer(max_features=20, ngram_range=(1,2))
brand_mat = brand_vec.fit_transform([preprocess(s) for s in taglines])
term_scores = brand_mat.sum(axis=0).A1
terms = brand_vec.get_feature_names_out()
sorted_idx = np.argsort(term_scores)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
colors_bar = ['#C9A84C' if i < 5 else '#3ECFB2' for i in range(len(terms))]
bars = ax.bar(terms[sorted_idx], term_scores[sorted_idx], color=colors_bar, alpha=0.85)
ax.set_title('Top TF-IDF Terms in Brand Slogan Corpus', color='#C9A84C', fontsize=13)
ax.set_xlabel('Term', color='white'); ax.set_ylabel('TF-IDF Score', color='white')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('assets/sample_exports/02_tfidf_terms.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ TF-IDF chart saved')

## Summary

| Step | Method | Output |
|---|---|---|
| Preprocessing | NLTK tokenisation + stopword removal | Clean token lists |
| Vectorisation | TF-IDF (max_features=500, ngrams 1-2) | Sparse 515×500 matrix |
| Retrieval | Cosine similarity | Top-k matching taglines |
| Analysis | Tone heuristics + lexical metrics | Brand voice profile |
| Generation | TF-IDF retrieval → Gemini creative expansion | 6 taglines per brand |

**Note:** The slogan corpus (15 brand slogans + 500 startup taglines) is small by design — it acts as a seed retrieval set, with Gemini API generating the creative expansions.

## 7. HuggingFace Sentence Transformers (Semantic Similarity)

> Uses `sentence-transformers` (HuggingFace) to compute semantic embeddings of slogans — goes beyond TF-IDF keyword matching to understand *meaning*.

In [ ]:
!pip install -q sentence-transformers 2>/dev/null
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import numpy as np

print("Loading HuggingFace all-MiniLM-L6-v2 model…")
model_hf = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all brand slogans + sample startup taglines
all_texts = taglines + startup_tags[:50]
print(f"Encoding {len(all_texts)} texts with sentence-transformers…")
embeddings = model_hf.encode(all_texts, show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")

# Semantic similarity retrieval
def semantic_retrieve(query: str, top_k: int = 5) -> list:
    q_emb = model_hf.encode([query])
    sims  = cos_sim(q_emb, embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [{'text': all_texts[i], 'score': round(float(sims[i]),4)} for i in top_idx]

# Compare TF-IDF vs Semantic retrieval
test_query = "innovative tech startup disrupting the market"
print(f'\nQuery: "{test_query}"')
print(f'\nTF-IDF results:')
for r in retrieve_slogans(test_query, 3):
    print(f"  [{r['score']:.4f}] {r['text']}")

print(f'\nSentence-Transformer (semantic) results:')
for r in semantic_retrieve(test_query, 3):
    print(f"  [{r['score']:.4f}] {r['text']}")

print('\nKey insight: Semantic search finds meaning-similar slogans even without keyword overlap.')
print('BrandSphere uses TF-IDF for speed + Gemini for creative expansion.')